In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

# --- 1. Load Data ---
print("--- 1. Loading Data ---")
RECEIVALS_FILE = "data/kernel/receivals.csv"
PURCHASE_ORDERS_FILE = "data/kernel/purchase_orders.csv"
MAPPING_FILE = "data/prediction_mapping.csv"
MATERIALS_FILE = "data/extended/materials.csv"
TRANSPORT_FILE = "data/extended/transportation.csv"

try:
    df_receivals = pd.read_csv(RECEIVALS_FILE)
    df_po = pd.read_csv(PURCHASE_ORDERS_FILE)
    df_mapping = pd.read_csv(MAPPING_FILE)
    df_materials = pd.read_csv(MATERIALS_FILE)
    df_transport = pd.read_csv(TRANSPORT_FILE)
    print("All data files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# --- 2. Perform All Cleaning Steps ---
print("\n--- 2. Cleaning Data ---")

# --- Clean Receivals ---
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
df_receivals_cleaned = df_receivals_cleaned[df_receivals_cleaned['net_weight'] > 0].copy()
print(f"df_receivals_cleaned ready. Shape: {df_receivals_cleaned.shape}")

# --- Clean Materials (Our mapping file) ---
material_map = df_materials[['product_id', 'rm_id', 'raw_material_format_type']].drop_duplicates()
product_to_rm_map = material_map.drop_duplicates(subset=['product_id'], keep='first')
print(f"product_to_rm_map ready. Shape: {product_to_rm_map.shape}")

# --- Clean Transport ---
df_transport_cleaned = df_transport[
    ['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name', 'net_weight']
].dropna(subset=['transporter_name']).copy()
df_transport_cleaned.rename(columns={'net_weight': 'transport_net_weight'}, inplace=True)
# Keep only the *last* transporter record for a given PO item
df_transport_cleaned = df_transport_cleaned.drop_duplicates(
    subset=['rm_id', 'purchase_order_id', 'purchase_order_item_no'], 
    keep='last'
)
print(f"df_transport_cleaned ready. Shape: {df_transport_cleaned.shape}")

# --- Clean Purchase Orders ---
df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce', utc=True)
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce', utc=True)
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit', 'created_date_time']).copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['quantity'] > 0].copy()
print(f"df_po_cleaned (before rm_id). Shape: {df_po_cleaned.shape}")

# --- Add rm_id AND material_format to PO table ---
df_po_cleaned = pd.merge(
    df_po_cleaned,
    product_to_rm_map,
    on='product_id',
    how='left' 
)
df_po_cleaned = df_po_cleaned.dropna(subset=['rm_id'])
print(f"df_po_cleaned (after rm_id). Shape: {df_po_cleaned.shape}")

# --- Clean Prediction Mapping (Just load) ---
print(f"df_mapping loaded. Shape: {df_mapping.shape}")

# --- 3. Create Validation Split ---
print("\n--- 3. Creating Validation Split ---")
VALIDATION_START_DATE = pd.to_datetime('2024-08-01', utc=True)
train_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] < VALIDATION_START_DATE].copy()
validation_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] >= VALIDATION_START_DATE].copy()
print("Training and validation sets created.")
print(f"Training receivals:       {len(train_receivals)}")
print(f"Validation receivals:     {len(validation_receivals)}")

print("\n--- Setup Complete (v3 Features) ---")

--- 1. Loading Data ---
All data files loaded successfully.

--- 2. Cleaning Data ---
df_receivals_cleaned ready. Shape: (122383, 10)
product_to_rm_map ready. Shape: (55, 3)
df_transport_cleaned ready. Shape: (27430, 5)
df_po_cleaned (before rm_id). Shape: (33113, 12)
df_po_cleaned (after rm_id). Shape: (31977, 14)
df_mapping loaded. Shape: (30450, 4)

--- 3. Creating Validation Split ---
Training and validation sets created.
Training receivals:       120268
Validation receivals:     2115

--- Setup Complete (v3 Features) ---


In [ ]:
print("--- Phase 4: Building Multi-Year Training Set (v3) ---")

def create_feature_set_v3(start_date_str, end_date_str):
    """
    Generates a complete (X, y) feature set for the given date range.
    INCLUDES NEW V3 FEATURES from all 4 files:
    - f_po_lead_time_days
    - f_po_status (one-hot)
    - f_material_format (one-hot)
    - f_transporter_name (one-hot)
    """
    print(f"\n--- Generating v3 feature set for {start_date_str} to {end_date_str} ---")
    
    # --- 1. Define Time Universe ---
    TRAIN_START_DATE = pd.to_datetime(start_date_str, utc=True)
    TRAIN_END_DATE = pd.to_datetime(end_date_str, utc=True)
    all_rm_ids = df_receivals_cleaned['rm_id'].unique()
    train_dates = pd.date_range(start=TRAIN_START_DATE, end=TRAIN_END_DATE, freq='D', tz='UTC')
    train_universe = list(itertools.product(all_rm_ids, train_dates))
    df = pd.DataFrame(train_universe, columns=['rm_id', 'forecast_end_date'])
    
    # --- 2. Define Historical Data for this Block ---
    hist_receivals = df_receivals_cleaned[df_receivals_cleaned['date_arrival'] < TRAIN_START_DATE].copy()
    
    # --- V3 CHANGE: Merge POs with Transport data ---
    hist_po = pd.merge(
        df_po_cleaned,
        df_transport_cleaned[['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name']],
        on=['rm_id', 'purchase_order_id', 'purchase_order_item_no'],
        how='left'
    )
    # Fill in a default transporter name
    hist_po['transporter_name'] = hist_po['transporter_name'].fillna('Transporter_Unknown')

    # --- 3. Build Target (y_cumulative_weight) ---
    print("Building target (y)...")
    daily_actuals = df_receivals_cleaned[
        (df_receivals_cleaned['date_arrival'] >= TRAIN_START_DATE) &
        (df_receivals_cleaned['date_arrival'] <= TRAIN_END_DATE)
    ]
    daily_actuals_grouped = daily_actuals.groupby(
        ['rm_id', daily_actuals['date_arrival'].dt.date]
    )['net_weight'].sum().reset_index(name='daily_net_weight')
    daily_actuals_grouped['forecast_end_date'] = pd.to_datetime(daily_actuals_grouped['date_arrival'], utc=True)
    df = pd.merge(
        df, daily_actuals_grouped[['rm_id', 'forecast_end_date', 'daily_net_weight']],
        on=['rm_id', 'forecast_end_date'], how='left'
    )
    df['daily_net_weight'] = df['daily_net_weight'].fillna(0)
    df = df.sort_values(by=['rm_id', 'forecast_end_date'])
    df['y_cumulative_weight'] = df.groupby('rm_id')['daily_net_weight'].cumsum()
    
    # --- 4. Build Aggregated PO Features (V3) ---
    print("Building aggregated PO features (v3)...")
    hist_po['f_po_lead_time_days'] = (hist_po['delivery_date'] - hist_po['created_date_time']).dt.days
    
    po_agg = hist_po.groupby(['rm_id', 'delivery_date']).agg(
        daily_po_quantity=('quantity', 'sum'),
        avg_lead_time=('f_po_lead_time_days', 'mean')
    ).reset_index()
    po_agg.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
    
    rm_ids_to_merge = df['rm_id'].unique()
    merged_df = pd.merge(
        df[['rm_id', 'forecast_end_date']],
        po_agg[po_agg['rm_id'].isin(rm_ids_to_merge)],
        on='rm_id', how='left'
    )
    merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
    cumulative_features = merged_df_filtered.groupby(['rm_id', 'forecast_end_date']).agg(
        f_cumulative_po_quantity=('daily_po_quantity', 'sum'),
        f_avg_lead_time=('avg_lead_time', 'mean')
    ).reset_index()
    df = pd.merge(df, cumulative_features, on=['rm_id', 'forecast_end_date'], how='left')
    df['f_cumulative_po_quantity'] = df['f_cumulative_po_quantity'].fillna(0)
    df['f_avg_lead_time'] = df['f_avg_lead_time'].fillna(0)
    
    # --- 5. Build Features (Time-Based) ---
    print("Building features (Time-Based)...")
    df['f_month'] = df['forecast_end_date'].dt.month
    df['f_day_of_week'] = df['forecast_end_date'].dt.dayofweek
    df['f_day_of_year'] = df['forecast_end_date'].dt.dayofyear
    df['f_is_month_end'] = df['forecast_end_date'].dt.is_month_end.astype(str) # For one-hot

    # --- 6. Build Features (Entity & Lag) ---
    print("Building features (Entity & Lag)...")
    
    # f_median_lag_days
    df_train_merged_eda = pd.merge(
        hist_receivals,
        hist_po[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
        on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'], how='inner'
    )
    df_train_merged_eda['delivery_lag_days'] = (
        df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
    ).dt.days
    rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
    df = pd.merge(df, rm_id_lag_map, on='rm_id', how='left')
    df['f_median_lag_days'] = df['f_median_lag_days'].fillna(0)

    # f_receivals_Nd
    hist_windows = [30, 90, 180]
    for days in hist_windows:
        hist_start_date_window = TRAIN_START_DATE - pd.Timedelta(days=days)
        window_data = hist_receivals[
            (hist_receivals['date_arrival'] >= hist_start_date_window) &
            (hist_receivals['date_arrival'] < TRAIN_START_DATE)
        ]
        feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=f'f_receivals_{days}d')
        df = pd.merge(df, feature_map, on='rm_id', how='left')
        df[f'f_receivals_{days}d'] = df[f'f_receivals_{days}d'].fillna(0)
        
    # --- V3: f_material_format, f_po_status, f_transporter_name ---
    # These are static features per rm_id (based on last known PO)
    rm_id_static_features = hist_po.drop_duplicates(subset=['rm_id'], keep='last')[[
        'rm_id', 'status', 'raw_material_format_type', 'transporter_name'
    ]]
    df = pd.merge(df, rm_id_static_features, on='rm_id', how='left')
    
    # Fill NaNs for rm_ids with no PO history
    df['status'] = df['status'].fillna('Unknown')
    df['raw_material_format_type'] = df['raw_material_format_type'].fillna('Unknown')
    df['transporter_name'] = df['transporter_name'].fillna('Unknown')

    print(f"--- Block for {start_date_str} complete. Shape: {df.shape} ---")
    
    # Filter to active rm_ids
    rm_id_max_y = df.groupby('rm_id')['y_cumulative_weight'].max()
    active_rm_ids = rm_id_max_y[rm_id_max_y > 0].index
    df = df[df['rm_id'].isin(active_rm_ids)]
    
    print(f"--- Filtered to active rm_ids. Final Shape: {df.shape} ---")
    return df

--- Phase 4: Building Multi-Year Training Set (v3) ---


In [ ]:
# --- Build Training Set ---
print("\n\n--- Building Full Training Set (v3) ---")
years = [2023, 2022, 2021, 2020]
training_blocks = []
for year in years:
    start_date = f'{year}-08-01'
    end_date = f'{year}-12-31'
    if pd.to_datetime(f'{year}-01-01', utc=True) > train_receivals['date_arrival'].min():
        block = create_feature_set_v3(start_date, end_date)
        training_blocks.append(block)
df_train = pd.concat(training_blocks, ignore_index=True)
print("\n--- Full Training Set (v3) Created ---")
df_train.info()

# --- Build Validation Set ---
print("\n\n--- Building Validation Set (v3) ---")
df_val = create_feature_set_v3('2024-08-01', '2024-12-19')
print("\n--- Validation Set (v3) Created ---")
df_val.info()

# --- One-Hot Encode Categoricals ---
print("\n\n--- One-Hot Encoding Categoricals ---")
CATEGORICAL_COLS = ['f_is_month_end', 'status', 'raw_material_format_type', 'transporter_name']
df_train = pd.get_dummies(df_train, columns=CATEGORICAL_COLS, prefix_sep='_f_')
df_val = pd.get_dummies(df_val, columns=CATEGORICAL_COLS, prefix_sep='_f_')

# Align columns - crucial if one set is missing a category
train_cols, val_cols = df_train.align(df_val, join='outer', axis=1, fill_value=0)
df_train = train_cols
df_val = val_cols

# --- Save Files ---
print("\n\n--- Saving v3 Files ---")
try:
    feature_columns = [col for col in df_train.columns if col.startswith('f_')]
    target_column = 'y_cumulative_weight'
    id_columns = ['rm_id', 'forecast_end_date']
    
    df_train_save = df_train[id_columns + feature_columns + [target_column]].copy()
    df_val_save = df_val[id_columns + feature_columns + [target_column]].copy()
    
    df_train_save.to_parquet("training_dataset_v3.parquet", index=False)
    df_val_save.to_parquet("validation_dataset_v3.parquet", index=False)
    
    print(f"Successfully saved 'training_dataset_v3.parquet' (Shape: {df_train_save.shape})")
    print(f"Successfully saved 'validation_dataset_v3.parquet' (Shape: {df_val_save.shape})")
    print("\n--- 03_Feature_Engineering_v3.ipynb Complete ---")

except Exception as e:
    print(f"\nError saving file: {e}")



--- Building Full Training Set (v3) ---

--- Generating v3 feature set for 2023-08-01 to 2023-12-31 ---
Building target (y)...
Building aggregated PO features (v3)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2023-08-01 complete. Shape: (31059, 17) ---
--- Filtered to active rm_ids. Final Shape: (7038, 17) ---

--- Generating v3 feature set for 2022-08-01 to 2022-12-31 ---
Building target (y)...
Building aggregated PO features (v3)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2022-08-01 complete. Shape: (31059, 17) ---
--- Filtered to active rm_ids. Final Shape: (5508, 17) ---

--- Generating v3 feature set for 2021-08-01 to 2021-12-31 ---
Building target (y)...
Building aggregated PO features (v3)...
Building features (Time-Based)...
Building features (Entity & Lag)...
--- Block for 2021-08-01 complete. Shape: (31059, 17) ---
--- Filtered to active rm_ids. Final Shape: (4590, 17) ---

--- Generating 